In [0]:
dbutils.widgets.text("catalog", "dev", "Catalog Name")
CATALOG = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import *

In [0]:
# ---- CONFIGURATION ----
BRONZE_SCHEMA = "bronze"
RAW_VOLUME_PATH = f"/Volumes/{CATALOG}/bronze/raw"

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

print(f"Catalog: {CATALOG}")
print(f"Schema: {BRONZE_SCHEMA}")
print(f"Raw files: {RAW_VOLUME_PATH}")

In [0]:
files = dbutils.fs.ls(RAW_VOLUME_PATH)
for f in files:
    print(f"  {f.name:30s} {f.size / (1024*1024):.2f} MB")

In [0]:
SOURCE_TABLES = {
    "customers": {
        "file": f"{RAW_VOLUME_PATH}/customers.csv",
        "target_table": "raw_customers",
        "primary_key": "customer_id",
    },
    "orders": {
        "file": f"{RAW_VOLUME_PATH}/orders.csv",
        "target_table": "raw_orders",
        "primary_key": "order_id",
    },
    "order_items": {
        "file": f"{RAW_VOLUME_PATH}/order_items.csv",
        "target_table": "raw_order_items",
        "primary_key": "order_item_id",
    },
    "products": {
        "file": f"{RAW_VOLUME_PATH}/products.csv",
        "target_table": "raw_products",
        "primary_key": "product_id",
    },
}

In [0]:
print(f"Registered {len(SOURCE_TABLES)} source tables:")
for name in SOURCE_TABLES:
    print(f"  - {name} → {SOURCE_TABLES[name]['target_table']}")

In [0]:
for name, config in SOURCE_TABLES.items():
    # print(f"\n{'='*60}")
    # print(f"  SOURCE: {name}")
    # print(f"{'='*60}")
    df = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .csv(config["file"]))

In [0]:

def ingest_to_bronze(source_name, config, mode="overwrite"):
    """
    Read a CSV source file and write it as a Delta table in Bronze layer.
    
    Adds audit columns:
      - _ingested_at   : timestamp when this row was ingested
      - _source_file   : original file path
      - _source_name   : logical source name
    
    Parameters:
      source_name : str - logical name (e.g., "customers")
      config      : dict - must have 'file' and 'target_table' keys
      mode        : str - "overwrite" for initial load, "append" for incremental
    """
    
    print(f"\n--- Ingesting: {source_name} ---")
    
    # 1. Read raw CSV (all columns as inferred types)
    df_raw = (spark.read
              .option("header", True)
              .option("inferSchema", True)
              .option("multiLine", True)
              .option("escape", '"')
              .csv(config["file"]))
    
    raw_count = df_raw.count()
    print(f"  Read {raw_count:,} rows from {config['file']}")
    
    # 2. Add audit/metadata columns (Bronze standard)
    df_bronze = (df_raw
                 .withColumn("_ingested_at", current_timestamp())
                 .withColumn("_source_file", df_raw["_metadata.file_path"])
                 .withColumn("_source_name", lit(source_name)))
    
    # 3. Write as Delta table
    target = f"{CATALOG}.{BRONZE_SCHEMA}.{config['target_table']}"
    
    (df_bronze.write
     .format("delta")
     .mode(mode)
     .option("overwriteSchema", "true")
     .saveAsTable(target))
    
    # 4. Verify
    written_count = spark.table(target).count()
    print(f"  Written {written_count:,} rows to {target}")
    print(f"  Status: {'SUCCESS' if written_count == raw_count else 'MISMATCH - CHECK!'}")
    
    return {"source": source_name, "target": target, "rows": written_count}


In [0]:
results = []
for name, config in SOURCE_TABLES.items():
    result = ingest_to_bronze(name, config, mode="overwrite")
    results.append(result)

In [0]:
print(f"\n{'='*60}")
print(f"  BRONZE INGESTION SUMMARY")
print(f"{'='*60}")
for r in results:
    print(f"  {r['source']:20s} → {r['target']:40s} | {r['rows']:>10,} rows")
print(f"{'='*60}")
print(f"  Total tables ingested: {len(results)}")

In [0]:
# print("Bronze Layer Tables:\n")

# for name, config in SOURCE_TABLES.items():
#     target = f"{CATALOG}.{BRONZE_SCHEMA}.{config['target_table']}"
#     df = spark.table(target)
#     print(f"  {config['target_table']}")
#     print(f"    Rows    : {df.count():,}")
#     print(f"    Columns : {df.columns}")
#     print()